# Time-Frequency Analysis — Chapter 6
# 時間周波数解析 — 第6章

## The Ambiguity Function and Its Role in Cohen's Class
## アンビギュイティ関数とCohenのクラスにおける役割

---
> **Reference:** Cohen, *Time-Frequency Analysis*, Ch. 5; Woodward (1953); Ville (1948).

## 6.1 Definition of the Ambiguity Function
## 6.1 アンビギュイティ関数の定義

**English.**  
The **symmetric ambiguity function (AF)** of a signal $x(t)$ is:

$$\boxed{A_x(\eta, \tau) = \int_{-\infty}^{\infty} x\!\left(t + \frac{\tau}{2}\right) x^*\!\left(t - \frac{\tau}{2}\right)\, e^{-j2\pi\eta t}\, dt}$$

- $\eta$ — **Doppler frequency** (frequency shift variable)
- $\tau$ — **time delay** (lag variable)

Originally introduced by Woodward (1953) in radar theory to characterise a waveform's ability to resolve targets differing in range ($\tau$) and velocity ($\eta$).

The ambiguity function is the **2-D Fourier transform** of the WVD:

$$A_x(\eta, \tau) = \iint W_x(t, f)\, e^{-j2\pi(\eta t - f\tau)}\, dt\, df$$

And conversely:

$$W_x(t, f) = \iint A_x(\eta, \tau)\, e^{j2\pi(\eta t - f\tau)}\, d\eta\, d\tau$$

---

**日本語.**  
信号 $x(t)$ の**対称アンビギュイティ関数（AF）** は以下で定義されます：

$$\boxed{A_x(\eta, \tau) = \int_{-\infty}^{\infty} x\!\left(t + \frac{\tau}{2}\right) x^*\!\left(t - \frac{\tau}{2}\right)\, e^{-j2\pi\eta t}\, dt}$$

- $\eta$ — **ドップラー周波数**（周波数シフト変数）
- $\tau$ — **時間遅延**（ラグ変数）

Woodward（1953年）がレーダー理論で導入しました。  
アンビギュイティ関数は WVD の**2次元フーリエ変換**です。

## 6.2 Properties of the Ambiguity Function
## 6.2 アンビギュイティ関数の性質

**English.**  

### Maximum at the origin
$$|A_x(\eta, \tau)| \leq A_x(0, 0) = \int |x(t)|^2\, dt = E$$

The AF always achieves its maximum at the **origin** — this is the fundamental theorem of radar ambiguity.

### Slices
- **Along $\eta = 0$ (zero Doppler):** $A_x(0, \tau) = \int x(t+\tau/2)x^*(t-\tau/2)dt$ — the autocorrelation of $x$
- **Along $\tau = 0$ (zero delay):** $A_x(\eta, 0) = \int |x(t)|^2 e^{-j2\pi\eta t}dt$ — the FT of the instantaneous power

### Volume invariant (Moyal)
$$\iint |A_x(\eta, \tau)|^2\, d\eta\, d\tau = E^2$$

The total volume of $|A_x|^2$ is conserved — spreading the ambiguity (better resolution in one direction) must come at the cost of concentration elsewhere.

### Cross-term location
For $x = x_1 + x_2$, the cross-ambiguity $A_{x_1 x_2}(\eta, \tau)$ is centred away from the auto-terms, **displaced** by $(\Delta\eta, \Delta\tau)$ corresponding to the TF separation of $x_1$ and $x_2$.  
This is why kernel suppression in the AF domain can cleanly separate auto-terms from cross-terms.

---

**日本語.**  

### 原点での最大値
$$|A_x(\eta, \tau)| \leq A_x(0, 0) = E$$

AF は常に**原点**で最大値を取ります — これがレーダーのアンビギュイティの基本定理です。

### 体積不変則（Moyal）
$$\iint |A_x(\eta, \tau)|^2\, d\eta\, d\tau = E^2$$

$|A_x|^2$ の総体積は保存されます。ある方向の分解能を改善すると、別の方向で犠牲が生じます。

### クロスタームの位置
$x = x_1 + x_2$ の場合、交差アンビギュイティ $A_{x_1 x_2}$ は自己タームから**ずれた位置**に現れます。これが、AFドメインでのカーネル抑制が自己タームとクロスタームをきれいに分離できる理由です。

## 6.3 The Fundamental Diagram: AF ↔ TF Plane
## 6.3 基本図式：AFドメイン ↔ 時間周波数平面

**English.**  
The relationship between the ambiguity function, the WVD, and Cohen's class can be summarised in a commutative diagram:

$$\begin{array}{ccc}
x(t) \cdot x^*(t) & \xrightarrow{\text{bilinear}} & \text{Instantaneous AC: } R_x(t,\tau)\\
\downarrow \mathcal{F}_t & & \downarrow \mathcal{F}_\tau \\
A_x(\eta, \tau) & \xrightarrow{\times\,\Phi(\eta,\tau)} & A_x \cdot \Phi \\
\downarrow \mathcal{F}_{\eta,\tau}^{-1} & & \downarrow \mathcal{F}_{\eta,\tau}^{-1} \\
W_x(t, f) & & C_x(t, f)
\end{array}$$

**Key insight:** Cohen's class member $C_x$ is obtained by **filtering $A_x$ with kernel $\Phi$** in the AF domain, then transforming back.  
Auto-terms → near origin in AF → pass through $\Phi$.  
Cross-terms → far from origin in AF → attenuated by $\Phi$.

---

**日本語.**  
アンビギュイティ関数、WVD、Cohenのクラスの関係を交換図式で表すと：

**重要な洞察：** Cohenのクラスのメンバー $C_x$ は、**AFドメインで $A_x$ をカーネル $\Phi$ でフィルタリング**してから逆変換することで得られます。  
- 自己項 → AFで原点付近 → $\Phi$ を通過  
- クロスターム → AFで原点から遠い → $\Phi$ で減衰

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})

In [ ]:
# ─── Discrete Ambiguity Function ──────────────────────────────────────────────
# 離散アンビギュイティ関数

def ambiguity_function_2d(z, N_fft=None):
    """
    Compute the symmetric ambiguity function |A_z(η, τ)|.
    Returns |A|[eta_idx, tau_idx], normalised η ∈ [-0.5, 0.5].
    アナリティック信号 z のアンビギュイティ関数を計算する。
    """
    N = len(z)
    if N_fft is None:
        N_fft = 2 * N

    # Instantaneous autocorrelation matrix R[t, τ]
    R = np.zeros((N, N), dtype=complex)
    for lag in range(N):
        t_valid = np.arange(lag, N - lag)
        if len(t_valid) == 0:
            break
        R[t_valid, lag] = z[t_valid + lag] * np.conj(z[t_valid - lag])
        if lag > 0:
            R[t_valid, N - lag] = np.conj(R[t_valid, lag])

    # A(η, τ): FT along the time axis (axis=0)
    A = np.fft.fftshift(np.fft.fft(R, n=N_fft, axis=0), axes=0)
    eta = np.fft.fftshift(np.fft.fftfreq(N_fft))
    tau = np.arange(N) / N  # normalised [0, 1)

    return A, eta, tau

print("Ambiguity function implemented / アンビギュイティ関数を実装しました")

In [ ]:
# ─── AF of a Gaussian atom ────────────────────────────────────────────────────
# ガウスアトムのアンビギュイティ関数

fs = 256
N  = 256
t  = np.arange(N) / fs

# Gaussian atom centred at t0=0.5s, f0=60 Hz, σ=0.05s
sigma_g = 0.05
t0, f0  = 0.5, 60.0
z_gauss = np.exp(-((t - t0)**2) / (2*sigma_g**2)) * np.exp(1j * 2*np.pi*f0*t)

N_fft = 512
A, eta, tau = ambiguity_function_2d(z_gauss, N_fft=N_fft)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Ambiguity Function of a Gaussian atom\n'
             'ガウスアトムのアンビギュイティ関数', fontsize=12)

# |A(η, τ)|
A_abs = np.abs(A[:, :N//2])
im = axes[0].pcolormesh(tau[:N//2], eta, A_abs,
                         cmap='viridis', shading='gouraud')
axes[0].set_xlabel('Lag τ (normalised)')
axes[0].set_ylabel('Doppler η (normalised)')
axes[0].set_title(r'$|A_x(\eta, \tau)|$')
plt.colorbar(im, ax=axes[0])

# Slices
zero_eta_idx = np.argmin(np.abs(eta))
zero_tau_idx = 0
axes[1].plot(tau[:N//2], A_abs[zero_eta_idx, :], 'b-', lw=2,
             label=r'$|A_x(0, \tau)|$ — autocorrelation')
axes[1].plot(tau[:N//2], A_abs[:, zero_tau_idx][:len(tau[:N//2])],
             'r--', lw=2, label=r'$|A_x(\eta, 0)|$ — power spectrum FT')
axes[1].set_xlabel('Lag / Doppler (normalised)')
axes[1].set_ylabel('|AF|')
axes[1].set_title('AF slices / AFのスライス')
axes[1].legend(fontsize=9)

# Confirm: maximum at origin
print(f"Max |A|  = {A_abs.max():.4f}")
print(f"|A(0,0)| = {A_abs[zero_eta_idx, 0]:.4f}")
print("Maximum is at origin? / 最大値は原点?:", 
      np.isclose(A_abs.max(), A_abs[zero_eta_idx, 0], rtol=0.01))

plt.tight_layout()
plt.savefig('../figures/06_af_gaussian.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── AF cross-terms: auto vs cross in ambiguity domain ────────────────────────
# AFにおけるクロスターム：アンビギュイティドメインでの自己項とクロスターム

fs = 256
N  = 256
t  = np.arange(N) / fs

sigma_g = 0.06
a1 = np.exp(-((t - 0.3)**2)/(2*sigma_g**2)) * np.exp(1j*2*np.pi*30*t)
a2 = np.exp(-((t - 0.7)**2)/(2*sigma_g**2)) * np.exp(1j*2*np.pi*70*t)
z_combined = a1 + a2

N_fft = 512
A1,  eta, tau = ambiguity_function_2d(a1, N_fft=N_fft)
A2,  _,   _   = ambiguity_function_2d(a2, N_fft=N_fft)
A12, _,   _   = ambiguity_function_2d(z_combined, N_fft=N_fft)

tau_half = tau[:N//2]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Ambiguity function: auto-terms vs cross-terms\n'
             'アンビギュイティ関数：自己項とクロスタームの分離', fontsize=12)

kw = dict(cmap='hot_r', shading='gouraud')

for ax, (title, A) in zip(axes, [
    ('Auto-terms only\n$|A_1 + A_2|$ (no cross)', np.abs(A1[:, :N//2]) + np.abs(A2[:, :N//2])),
    ('Combined signal\n$|A_{1+2}(\\eta,\\tau)|$', np.abs(A12[:, :N//2])),
    ('Cross-term contribution\n$|A_{12}| - (|A_1|+|A_2|)$',
     np.abs(A12[:, :N//2]) - np.abs(A1[:, :N//2]) - np.abs(A2[:, :N//2])),
]):
    vmax = max(np.abs(A).max(), 1e-3)
    im = ax.pcolormesh(tau_half, eta, A, vmin=0, vmax=vmax, **kw)
    ax.set_xlabel('Lag τ')
    ax.set_ylabel('Doppler η') if ax == axes[0] else None
    ax.set_title(title, fontsize=9)
    ax.set_ylim(-0.5, 0.5)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig('../figures/06_af_crossterms.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── Radar interpretation: ambiguity function of LFM pulse ────────────────────
# レーダー解釈：LFM パルスのアンビギュイティ関数

fs = 512
N  = 256
t  = np.arange(N) / fs

# Linear frequency-modulated (LFM / chirp) pulse — standard radar waveform
# 線形周波数変調（LFM）パルス — 標準的なレーダー波形
B = 100.0   # bandwidth [Hz] / 帯域幅
T_pulse = N / fs
x_lfm = np.where((t >= 0) & (t < T_pulse),
                  np.cos(2*np.pi * (20*t + B/(2*T_pulse)*t**2)),
                  0.0)
z_lfm = hilbert(x_lfm)

N_fft = 512
A_lfm, eta, tau = ambiguity_function_2d(z_lfm, N_fft=N_fft)
A_mag = np.abs(A_lfm[:, :N//2])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LFM pulse: ambiguity function (radar waveform analysis)\n'
             'LFM パルスのアンビギュイティ関数（レーダー波形解析）', fontsize=12)

im = axes[0].pcolormesh(tau[:N//2], eta, A_mag / A_mag.max(),
                         cmap='jet', vmin=0, vmax=1, shading='gouraud')
axes[0].set_xlabel('Range (delay τ)')
axes[0].set_ylabel('Velocity (Doppler η)')
axes[0].set_title('|A(η, τ)| (normalised)\nLFM "ridge" = range-Doppler coupling')
plt.colorbar(im, ax=axes[0])

# Zero-Doppler cut: pulse compression
zero_eta = np.argmin(np.abs(eta))
axes[1].plot(tau[:N//2], A_mag[zero_eta, :] / A_mag[zero_eta, 0],
             'b-', lw=2)
axes[1].set_xlabel('Lag τ')
axes[1].set_ylabel('Normalised |A(0, τ)|')
axes[1].set_title('Zero-Doppler cut (pulse compression)\nゼロドップラースライス（パルス圧縮）')
axes[1].axvline(0, color='r', ls='--', lw=1)

plt.tight_layout()
plt.savefig('../figures/06_af_lfm_radar.png', bbox_inches='tight')
plt.show()

print("LFM ambiguity function: the 'knife-edge' ridge shows")
print("the range-Doppler coupling inherent in chirp waveforms.")
print("LFM のアンビギュイティ関数：『刃』状のリッジは")
print("チャープ波形に固有のレンジ-ドップラー結合を示す。")

## 6.4 Summary
## 6.4 まとめ

| Property | Formula / Result |
|----------|------------------|
| Definition | $A_x(\eta,\tau) = \int x(t+\tau/2)x^*(t-\tau/2)e^{-j2\pi\eta t}dt$ |
| Relationship to WVD | $A_x = \mathcal{F}_{2D}\{W_x\}$ |
| Maximum at origin | $|A_x(\eta,\tau)| \leq A_x(0,0) = E$ |
| Volume invariance | $\iint |A_x|^2\,d\eta\,d\tau = E^2$ |
| Cohen's class | $C_x = \mathcal{F}^{-1}_{2D}\{A_x \cdot \Phi\}$ |

**English.**  
The ambiguity function is the **natural domain for kernel design** in Cohen's class.  
Auto-terms cluster near the AF origin (large $|A|$) while cross-terms are pushed away — making a radially-decaying kernel $\Phi$ the ideal cross-term suppressor.  
In radar and sonar, the AF directly quantifies a waveform's resolution cell in range and Doppler.

---

**日本語.**  
アンビギュイティ関数はCohenのクラスにおける**カーネル設計の自然なドメイン**です。  
自己項はAFの原点付近に集中し（大きな $|A|$）、クロスタームは遠くに押し出されます。これにより、半径方向に減衰するカーネル $\Phi$ が理想的なクロスターム抑制器となります。  
レーダーや水中音響では、AFは波形のレンジおよびドップラー分解能セルを直接定量化します。